# ERCOT Decarbonization Scenario

This example demonstrates a simplified 4-zone ERCOT-style US regional grid scenario targeting 80% renewables by 2035 with carbon pricing.

**Key features:**
- 4 ERCOT weather zones: Houston, North, South, West
- Existing thermal fleet (gas CCGT, gas CT, coal, nuclear)
- Extendable renewables (onshore wind, solar)
- Battery storage (4-hour) in each zone
- Long-duration hydrogen storage in Houston
- Inter-zonal transmission links (partially extendable)
- CO2 emission cap of 50 Mt
- 80% renewable energy target as a custom constraint

The network uses 2920 hourly snapshots (every 3rd hour of a year) for tractability.

In [ ]:
import pypsa
import matplotlib.pyplot as plt
import pandas as pd

## Load the Example Network

In [ ]:
n = pypsa.examples.ercot_decarbonization()
print(n)

## Inspect the Network

### Carriers and CO2 Emissions

In [ ]:
n.carriers[["co2_emissions"]]

### Existing Generation Fleet

In [ ]:
existing = n.generators[~n.generators.p_nom_extendable]
existing[["bus", "carrier", "p_nom", "marginal_cost", "efficiency"]]

### Extendable Renewable Generators

In [ ]:
extendable = n.generators[n.generators.p_nom_extendable]
extendable[["bus", "carrier", "capital_cost", "p_nom_max"]]

### Storage Assets

In [ ]:
print("Battery Storage Units:")
print(n.storage_units[["bus", "carrier", "capital_cost", "max_hours"]])
print("\nHydrogen Store:")
print(n.stores[["bus", "capital_cost", "e_nom_extendable"]])
print("\nHydrogen Links (electrolyzer + fuel cell):")
h2_links = n.links[n.links.carrier == "hydrogen"]
print(h2_links[["bus0", "bus1", "efficiency", "capital_cost"]])

### Inter-zonal Transmission Links

In [ ]:
tx_links = n.links[n.links.carrier == "AC"]
tx_links[["bus0", "bus1", "p_nom", "p_nom_extendable", "capital_cost"]]

### CO2 Constraint

In [ ]:
n.global_constraints

## Add 80% Renewable Target as Custom Constraint

We add an extra constraint requiring that at least 80% of total demand is met by renewable generation (onshore wind and solar).

In [ ]:
def renewable_target(n, sns):
    """Require at least 80% of total demand from renewables."""
    m = n.model
    total_demand = n.loads_t.p_set.loc[sns].sum().sum()
    re_carriers = ["onshore_wind", "solar"]
    re_gens = n.generators.index[n.generators.carrier.isin(re_carriers)]
    weights = n.snapshot_weightings.generators.loc[sns]
    re_production = (m.variables["Generator-p"].sel(name=re_gens) * weights).sum()
    m.add_constraints(
        re_production >= 0.8 * total_demand, name="renewable_share_80pct"
    )

## Run the Optimization

In [ ]:
status, condition = n.optimize(
    solver_name="highs",
    extra_functionality=renewable_target,
)
print(f"Status: {status}, Condition: {condition}")
print(f"Total system cost: ${n.objective:,.0f}")

## Analyze Results

### New Capacity Investments

In [ ]:
# Show optimal capacity for extendable assets
print("Generator capacity expansion (MW):")
print(n.generators.loc[n.generators.p_nom_extendable, ["bus", "carrier", "p_nom_opt"]])
print("\nStorage capacity expansion (MW):")
print(n.storage_units.loc[n.storage_units.p_nom_extendable, ["bus", "p_nom_opt", "max_hours"]])
print("\nTransmission expansion (MW):")
print(n.links.loc[n.links.p_nom_extendable, ["bus0", "bus1", "carrier", "p_nom", "p_nom_opt"]])

### Energy Balance

In [ ]:
n.statistics.energy_balance()

### CO2 Shadow Price

In [ ]:
print("CO2 constraint shadow price ($/tCO2):")
print(n.global_constraints.mu)

### Sample Week Dispatch

Plot the dispatch for a representative summer week to visualize the generation mix.

In [ ]:
# Select a summer week
week_start = "2035-07-01"
week_end = "2035-07-08"
dispatch = n.generators_t.p.loc[week_start:week_end]

fig, ax = plt.subplots(figsize=(14, 6))
dispatch.plot.area(ax=ax, linewidth=0)
ax.set_ylabel("Generation (MW)")
ax.set_xlabel("Time")
ax.set_title("Generator Dispatch — Summer Week")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()

### Storage State of Charge

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Battery state of charge
soc = n.storage_units_t.state_of_charge.loc[week_start:week_end]
soc.plot(ax=axes[0])
axes[0].set_ylabel("State of Charge (MWh)")
axes[0].set_title("Battery Storage — State of Charge")

# H2 store state of charge
if not n.stores_t.e.empty:
    n.stores_t.e.loc[week_start:week_end].plot(ax=axes[1])
    axes[1].set_ylabel("Energy Stored (MWh)")
    axes[1].set_title("Hydrogen Store — Energy Level")

plt.tight_layout()

## Summary

This example demonstrates capacity expansion planning with:
- **CO2 cap**: Limits total emissions to 50 Mt
- **80% renewable target**: Custom constraint requiring renewables to meet 80% of demand
- **Multi-zone transmission**: Inter-zonal links that can be expanded
- **Storage mix**: Short-duration batteries and long-duration hydrogen storage

The optimization determines the least-cost investment and dispatch strategy that satisfies both the CO2 cap and the renewable energy target.